In [4]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_DIR = "/home/psxkf4/panda_phypush/csv_data/inference"
OUTPUT_DIR = "comparison_results_real"

# Mapping readable names to the keys found in your _metadata.csv files
METRICS = {
    "Mass Error (MAE)": "MASS_MAE",
    "Mass Error (nMAE %)": "MASS_NMAE_PCT",
    "Mass Error (NRMSE %)": "MASS_NRMSE_PCT",
    "Mass Error (sMAPE %)": "MASS_SMAPE_PCT",
    "Fric Error (MAE)": "MU_MAE",
    "Fric Error (nMAE %)": "MU_NMAE_PCT",
    "Fric Error (NRMSE %)": "MU_NRMSE_PCT",
    "Fric Error (sMAPE %)": "MU_SMAPE_PCT"
}

# ==========================================
# PROCESSING SCRIPT
# ==========================================

def load_inference_data(input_dir):
    """Crawls the inference directory and parses all _metadata.csv files."""
    search_pattern = os.path.join(input_dir, "**", "*_metadata.csv")
    files = glob.glob(search_pattern, recursive=True)
    
    all_data = []
    print(f"🔍 Found {len(files)} metadata files. Parsing...")
    
    for f in files:
        try:
            df = pd.read_csv(f)
            meta_dict = dict(zip(df['Parameter'], df['Value']))
            
            # Construct a 'domain' string
            cond = meta_dict.get('CONDITION', 'Unknown')
            mgt = meta_dict.get('MGT', 'Unknown')
            win = meta_dict.get('SMOOTHING_WINDOW', 'Unknown')
            domain = f"{cond} (mgt:{mgt}, w:{win})"
            
            row = {
                'domain': domain,
                'Model': meta_dict.get('VERSION_TAG', 'Unknown')
            }
            
            # Extract requested metrics safely
            for readable_name, col_key in METRICS.items():
                val = meta_dict.get(col_key, np.nan)
                if val == 'N/A' or pd.isna(val):
                    val = np.nan
                row[col_key] = float(val) if pd.notnull(val) else np.nan
                
            all_data.append(row)
        except Exception as e:
            print(f"❌ Error reading {f}: {e}")
            
    df_results = pd.DataFrame(all_data)
    
    # --- NEW: KEEP ONLY THE BEST RUN ---
    # 1. Sort by the primary error metric (Mass nMAE %) ascending (lowest error first)
    df_results = df_results.sort_values(by=['domain', 'Model', 'MASS_NMAE_PCT'], ascending=True)
    
    # 2. Drop duplicates, keeping the first row (which is now guaranteed to be the lowest error)
    original_count = len(df_results)
    df_results = df_results.drop_duplicates(subset=['domain', 'Model'], keep='first')
    new_count = len(df_results)
    
    if original_count != new_count:
        print(f"🧹 Filtered out {original_count - new_count} duplicate runs (Kept the lowest Mass Error).")
        
    return df_results

def save_wide_summary(df, output_dir, model_order):
    """Pivots the dataframe so Models are columns, and saves to CSV."""
    if df.empty: return []
    
    domain_order = sorted(df['domain'].unique().tolist())
    generated_files = []

    for readable_name, col in METRICS.items():
        if col not in df.columns or df[col].isna().all():
            continue
            
        clean_label = col.lower()
        
        # Create wide format
        wide = df.pivot_table(index='domain', columns='Model', values=col, aggfunc='first', observed=False)
        
        # Filter orders based on what actually exists in this pivot
        existing_indices = [d for d in domain_order if d in wide.index]
        existing_cols = [m for m in model_order if m in wide.columns]
        
        wide = wide.reindex(index=existing_indices, columns=existing_cols)
        
        csv_name = f"{clean_label}_wide.csv"
        csv_path = os.path.join(output_dir, csv_name)
        wide.to_csv(csv_path)
        
        generated_files.append((csv_name, readable_name))
        
    return generated_files

def csv_to_latex(input_csv, output_tex, model_names, caption="Performance Comparison"):
    """Converts the Wide CSV into a beautifully formatted LaTeX table."""
    if not os.path.exists(input_csv): return
    df = pd.read_csv(input_csv)
    
    # Escape LaTeX special characters
    safe_caption = caption.replace('%', r'\%').replace('_', r'\_')
    
    # Only use models that exist in the CSV columns
    csv_col_names = [m for m in model_names if m in df.columns]
    
    headers_top = ["\\textbf{Domain}"]
    
    for name in csv_col_names:
        latex_safe_name = str(name).strip().replace('_', r'\_')
        headers_top.append(f"\\textbf{{{latex_safe_name}}}")

    col_layout = "l " + "c" * (len(headers_top) - 1)
    
    latex = [
        "\\begin{table}[H]", # <-- CHANGED from [htbp] to [H] to stop floating overlaps
        "    \\centering",
        f"    \\caption{{{safe_caption}}}", 
        "    \\vspace{2mm}",
        "    \\resizebox{\\textwidth}{!}{",
        f"    \\begin{{tabular}}{{{col_layout}}}",
        "        \\toprule",
        "        " + " & ".join(headers_top) + " \\\\",
        "        \\midrule"
    ]

    for _, row in df.iterrows():
        domain_str = str(row['domain']).replace('_', r'\_')
        vals = [float(row[m]) if pd.notnull(row[m]) else np.nan for m in csv_col_names]
        
        if all(np.isnan(v) for v in vals): 
            continue
            
        # Highlight the best (minimum error) value in bold
        min_val = np.nanmin(vals)
        formatted_vals = []
        for v in vals:
            if np.isnan(v):
                formatted_vals.append("-")
            elif abs(v - min_val) < 1e-6:
                formatted_vals.append(f"\\textbf{{{v:.3f}}}")
            else:
                formatted_vals.append(f"{v:.3f}")
                
        latex.append(f"        {domain_str} & " + " & ".join(formatted_vals) + " \\\\")

    latex.extend([
        "        \\bottomrule", 
        "    \\end{tabular}", 
        "    }", 
        "\\end{table}"
    ])
    
    with open(output_tex, "w") as f:
        f.write("\n".join(latex) + "\n")

if __name__ == "__main__":
    # Ensure OUTPUT_DIR exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 1. Load Data
    df_results = load_inference_data(INPUT_DIR)
    
    if not df_results.empty:
        # Get unique models and sort them logically
        active_model_list = sorted(df_results['Model'].unique().tolist())
        print(f"📊 Models found: {active_model_list}")
        
        # 2. Save Wide CSVs
        tex_tasks = save_wide_summary(df_results, OUTPUT_DIR, active_model_list)
        
        # 3. Generate LaTeX Tables
        for csv_filename, readable_name in tex_tasks:
            csv_path = os.path.join(OUTPUT_DIR, csv_filename)
            tex_path = os.path.join(OUTPUT_DIR, csv_filename.replace(".csv", ".tex"))
            
            if os.path.exists(csv_path):
                csv_to_latex(csv_path, tex_path, active_model_list, caption=readable_name)
                print(f"📄 Generated LaTeX: {tex_path}")

        # 4. Concatenate into one master file with extra spacing
        master_tex_path = os.path.join(OUTPUT_DIR, "full_performance_report.tex")
        with open(master_tex_path, "w") as master_f:
            for i, (csv_filename, _) in enumerate(tex_tasks):
                tex_filename = csv_filename.replace(".csv", ".tex")
                tex_path = os.path.join(OUTPUT_DIR, tex_filename)
                
                if os.path.exists(tex_path):
                    with open(tex_path, "r") as f:
                        master_f.write(f.read())
                        master_f.write("\n\n% --- End of Table --- %\n\n")
                        
                        # Add clearpage between every single table to enforce spacing
                        master_f.write("\\vspace{1cm}\n\n\\clearpage\n\n")

        print(f"\n✅ All tasks completed. You can copy the code directly from:")
        print(f"📁 {master_tex_path}")
    else:
        print("❌ No data found! Double check the INPUT_DIR path.")

🔍 Found 72 metadata files. Parsing...
🧹 Filtered out 6 duplicate runs (Kept the lowest Mass Error).
📊 Models found: ['v1_data', 'v5_pinn', 'v6_pinn']
📄 Generated LaTeX: comparison_results_real/mass_mae_wide.tex
📄 Generated LaTeX: comparison_results_real/mass_nmae_pct_wide.tex
📄 Generated LaTeX: comparison_results_real/mass_nrmse_pct_wide.tex
📄 Generated LaTeX: comparison_results_real/mass_smape_pct_wide.tex

✅ All tasks completed. You can copy the code directly from:
📁 comparison_results_real/full_performance_report.tex
